# Multiple Linear Regression

Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

Generating Data

In [2]:
X, y = make_regression(
    n_samples=100,
    n_features=5,
    noise=10,
    random_state=42
)

data = pd.DataFrame(X, columns=[f'X{i}' for i in range(X.shape[1])])
data['y'] = y

print(data.head())

         X0        X1        X2        X3        X4           y
0  0.975120 -0.677162 -0.012247 -0.897254  0.075805  -72.840254
1  0.081874 -0.485364  0.758969 -0.772825 -0.236819  -56.769505
2 -1.412304 -0.908024 -0.562288 -1.012831  0.314247 -264.330770
3 -0.645120  0.361636  1.356240 -0.072010  1.003533  118.559167
4 -0.622700  0.280992 -1.952088 -0.151785  0.588317 -120.196006


EDA

In [3]:
data.shape

(100, 6)

In [4]:
print("X")
for col in data.columns:
    if col.startswith('X'):
        print(data[[col]].head())
print()
print(data[[col for col in data.columns if col.startswith('X')]].info())
print()
print(data[[col for col in data.columns if col.startswith('X')]].describe())

# Print y Target
print("\ny")
print(data[['y']].head())
print()
data[['y']].info()
print()
print(data['y'].describe())

X
         X0
0  0.975120
1  0.081874
2 -1.412304
3 -0.645120
4 -0.622700
         X1
0 -0.677162
1 -0.485364
2 -0.908024
3  0.361636
4  0.280992
         X2
0 -0.012247
1  0.758969
2 -0.562288
3  1.356240
4 -1.952088
         X3
0 -0.897254
1 -0.772825
2 -1.012831
3 -0.072010
4 -0.151785
         X4
0  0.075805
1 -0.236819
2  0.314247
3  1.003533
4  0.588317

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X0      100 non-null    float64
 1   X1      100 non-null    float64
 2   X2      100 non-null    float64
 3   X3      100 non-null    float64
 4   X4      100 non-null    float64
dtypes: float64(5)
memory usage: 4.0 KB
None

               X0          X1          X2          X3          X4
count  100.000000  100.000000  100.000000  100.000000  100.000000
mean    -0.032202    0.128058   -0.044747    0.076954   -0.093874
std      1.072806    0.959556  

In [5]:
for col in data.columns:
    if col.startswith('X'):
        print(data[[col]].isnull().sum())

print("y", data['y'].isnull().sum())

X0    0
dtype: int64
X1    0
dtype: int64
X2    0
dtype: int64
X3    0
dtype: int64
X4    0
dtype: int64
y 0


Model (Class)

In [6]:
class MultipleLinearRegression:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.coefficients = None
        self.intercept = None

    def fit(self):
        n_samples, n_features = self.X.shape
        X_b = np.hstack([np.ones((n_samples, 1)), self.X])
        theta_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(self.y)
        self.intercept = theta_best[0]
        self.coefficients = theta_best[1:]
    
    def predict(self, X):
        return self.intercept + X.dot(self.coefficients)

Training The Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(data[[col for col in data.columns if col.startswith('X')]], data['y'], test_size=0.2, random_state=42)
model = MultipleLinearRegression(X_train, y_train)
model.fit()
predictions = model.predict(X_test)

Evaluation Metrics (Class)

In [8]:
class MeanAbsoluteError:
    def __init__(self, y, y_pred):
        self.y = y
        self.y_pred = y_pred

    def compute(self):
        n = len(self.y)
        if n != len(self.y_pred):
            raise ValueError("y And y_pred Must Have The Same Length")
        
        error_sum = sum(abs(yi - ypi) for yi, ypi in zip(self.y, self.y_pred))
        mae = error_sum / n
        return mae

class MeanSquaredError:
    def __init__(self, y, y_pred):
        self.y = y
        self.y_pred = y_pred

    def compute(self):
        n = len(self.y)
        if n != len(self.y_pred):
            raise ValueError("y And y_pred Must Have The Same Length")
        
        error_sum = sum((yi - ypi)**2 for yi, ypi in zip(self.y, self.y_pred))
        mse = error_sum / n
        return mse

class RootMeanSquaredError:
    def __init__(self, y, y_pred):
        self.y = y
        self.y_pred = y_pred

    def compute(self):
        n = len(self.y)
        if n != len(self.y_pred):
            raise ValueError("y And y_pred Must Have The Same Length")
        
        error_sum = sum((yi - ypi)**2 for yi, ypi in zip(self.y, self.y_pred))
        rmse = np.sqrt(error_sum / n)
        return rmse

class R2_Score:
    def __init__(self, y, y_pred):
        self.y = y
        self.y_pred = y_pred
    
    def compute(self):
        if len(self.y) != len(self.y_pred):
            raise ValueError("y And y_pred Must Have The Same Length")
        
        y_mean = np.mean(self.y)
        
        numerator = sum((yi - ypi)**2 for yi, ypi in zip(self.y, self.y_pred))
        denominator = sum((yi - y_mean)**2 for yi in self.y)

        r2 = 1 - (numerator / denominator)
        return r2

import numpy as np

class Adjusted_R2_Score:
    def __init__(self, y, y_pred, p):
        self.y = y
        self.y_pred = y_pred
        self.n = len(y)
        self.p = p

    def compute(self):
        if len(self.y) != len(self.y_pred):
            raise ValueError("y And y_pred Must Have The Same Length")
        
        y_mean = np.mean(self.y)
        ss_total = sum((yi - y_mean)**2 for yi in self.y)
        ss_residual = sum((yi - ypi)**2 for yi, ypi in zip(self.y, self.y_pred))
        r2 = 1 - (ss_residual / ss_total)
        
        r2_adj = 1 - ((1 - r2) * (self.n - 1) / (self.n - self.p - 1))
        return r2_adj

Evaluation

In [9]:
mae = MeanAbsoluteError(y_test, predictions)
mse = MeanSquaredError(y_test, predictions)
rmse = RootMeanSquaredError(y_test, predictions)
r2_score = R2_Score(y_test, predictions)

print("Mean Absolute Error:", mae.compute())
print("Mean Squared Error:", mse.compute())
print("Root Mean Squared Error:", rmse.compute())
print("R^2 Score:", r2_score.compute())

Mean Absolute Error: 8.437614476433875
Mean Squared Error: 113.44800318060791
Root Mean Squared Error: 10.651197265125077
R^2 Score: 0.9944099738323338
